**Load features**

In [1]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torchvision import models
from torch.utils.data import Dataset
from torchvision import transforms
from PIL import Image
from torch.utils.data import DataLoader
import gc 
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

In [2]:
train_features = np.load("features\\train_data.npy")
test_features = np.load("features\\test_data.npy")
val_features = np.load("features\\val_data.npy")

train_labels = np.load("features\\train_labels.npy")
test_labels = np.load("features\\test_labels.npy")
val_labels = np.load("features\\val_labels.npy")

# Transform
encoder = OneHotEncoder(sparse_output=False)
train_labels = encoder.fit_transform(train_labels.reshape(-1, 1))
test_labels = encoder.transform(test_labels.reshape(-1, 1))
val_labels = encoder.transform(val_labels.reshape(-1, 1))

standardScaler = StandardScaler()
train_features = standardScaler.fit_transform(train_features)
test_features = standardScaler.transform(test_features)
val_features = standardScaler.transform(val_features)

print(train_features.shape)
print(test_features.shape)
print(val_features.shape)
print(train_labels.shape)
print(test_labels.shape)
print(val_labels.shape)

(4140, 162)
(143, 162)
(259, 162)
(4140, 8)
(143, 8)
(259, 8)


**Neural network architecture and training:**

In [ ]:
def buildResnet(num_classes):
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

    for param in model.parameters():
        param.requires_grad = False

    # Unfreeze 2 layer cuối để fine-tune
    for param in model.layer3.parameters():
        param.requires_grad = True
    for param in model.layer4.parameters():
        param.requires_grad = True

    model.fc = nn.Sequential(
        nn.Dropout(0.5), nn.Linear(model.fc.in_features, num_classes)
    )

    return model


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [12]:
class ImageDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, torch.tensor(self.labels[idx], dtype=torch.long)

In [27]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

val_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

**Upload processed spectrogram images & create DataLoader**

In [28]:
# Encode labels thành số
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

# Load CSV ảnh
train_csv = pd.read_csv("CSVs\\train_images.csv")
val_csv = pd.read_csv("CSVs\\val_images.csv")
test_csv = pd.read_csv("CSVs\\test_images.csv")

# Fit encoder trên train, transform tất cả
le.fit(train_csv["emotion"])
train_labels = le.transform(train_csv["emotion"])
val_labels = le.transform(val_csv["emotion"])
test_labels = le.transform(test_csv["emotion"])

# Tạo Dataset
train_dataset = ImageDataset(
    train_csv["path"].tolist(), train_labels, transform=train_transform
)
val_dataset = ImageDataset(
    val_csv["path"].tolist(), val_labels, transform=val_test_transform
)
test_dataset = ImageDataset(
    test_csv["path"].tolist(), test_labels, transform=val_test_transform
)

# Tạo DataLoader
train_loader = DataLoader(train_dataset, batch_size=32, pin_memory=True, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, pin_memory=True, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, pin_memory=True, shuffle=False)

print(
    f"Train samples: {len(train_dataset)}, Val samples: {len(val_dataset)}, Test samples: {len(test_dataset)}"
)
print("Classes:", list(le.classes_))

Train samples: 4140, Val samples: 259, Test samples: 143
Classes: ['Angry', 'Calm', 'Disgust', 'Fearful', 'Happy', 'Neutral', 'Sad', 'Surprised']


In [29]:
def evaluate(loader, model, criterion):
    model.eval()

    y_true, y_pred = [], []
    total_loss = 0

    with torch.no_grad():
        for images, emotions in loader:
            images = images.to(device)
            emotions = emotions.to(device)

            output = model(images)

            loss = criterion(output, emotions)
            total_loss += loss.item()

            preds = torch.argmax(output, dim=1).cpu().numpy()

            y_true.extend(emotions.cpu().numpy())
            y_pred.extend(preds)

    avg_loss = total_loss / len(loader)
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average="weighted")
    prec = precision_score(y_true, y_pred, average="weighted", zero_division=0)
    rec = recall_score(y_true, y_pred, average="weighted", zero_division=0)

    return avg_loss, acc, f1, prec, rec

## TRAIN / VALIDATE

In [ ]:
if "resnet" in dir():
    del resnet
    torch.cuda.empty_cache()
    gc.collect()

EPOCH = 20
LR = 1e-4

resnet = buildResnet(num_classes=8).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(resnet.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=3
)

best_val_loss = float("inf")
patience = 5
no_improve = 0

for epoch in range(EPOCH):
    train_loss = 0
    resnet.train()
    for images, emotions in train_loader:
        optimizer.zero_grad()
        output = resnet(images.to(device))
        loss = criterion(output, emotions.to(device))
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)
    print(f"Train Loss: {train_loss:.4f}")  # ← thêm dòng này

    val_loss, val_acc, val_f1, val_prec, val_rec = evaluate(
        val_loader, resnet, criterion
    )

    scheduler.step(val_loss)

    print(
        f"Epoch {epoch + 1}/{EPOCH} - Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}, "
        f"Val F1: {val_f1:.4f}, Val Prec: {val_prec:.4f}, Val Rec: {val_rec:.4f}"
    )

In [37]:
train_paths = set(train_csv["path"].tolist())
val_paths = set(val_csv["path"].tolist())
print(f"Ảnh trùng train/val: {len(train_paths & val_paths)}")

Ảnh trùng train/val: 0
